# Setup

In [ ]:
!pip install tqdm

In [ ]:
from os import environ
environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [ ]:
import os
from ckipnlp.pipeline import CkipPipeline, CkipDocument
pipeline = CkipPipeline(opts={'con_parser': {'username': os.environ["CKIP_USERNAME"], 'password': os.environ["CKIP_PASSWORD"]}})
from os import listdir
from tqdm import tqdm

In [ ]:
def flatten(l):
    return [item for sublist in l for item in sublist]

In [ ]:
paths=["../files/topics/nchu-topic/",
       "../files/topics/nkust-topic/",
       "../files/topics/ntnu-topic/",
       "../files/topics/nuk-topic/",
      ]
#tag_titles=['A', 'Caa', 'Cab', 'Cba', 'Cbb', 'D', 'Da', 'Dfa', 'Dfb', 'Di', 'Dk', 'DM', 'I', 'Na', 'Nb', 'Nc', 'Ncd', 'Nd', 'Nep', 'Neqa', 'Neqb', 'Nes', 'Neu', 'Nf', 'Ng', 'Nh', 'Nv', 'P', 'T', 'VA', 'VAC', 'VB', 'VC', 'VCL', 'VD', 'VF', 'VE', 'VG', 'VH', 'VHC', 'VI', 'VJ', 'VK', 'VL', 'V_2', 'DE', 'SHI', 'FW', 'COLONCATEGORY', 'COMMACATEGORY', 'DASHCATEGORY', 'DOTCATEGORY', 'ETCCATEGORY', 'EXCLAMATIONCATEGORY', 'PARENTHESISCATEGORY', 'PAUSECATEGORY', 'PERIODCATEGORY', 'QUESTIONCATEGORY', 'SEMICOLONCATEGORY', 'SPCHANGECATEGORY', 'WHITESPACE']
#tag_map={"A":0,"Caa":1,"Cab":2,"Cba":3,"Cbb":4,"D":5,"Da":6,"Dfa":7,"Dfb":8,"Di":9,"Dk":10,"DM":11,"I":12,"Na":13,"Nb":14,"Nc":15,"Ncd":16,"Nd":17,"Nep":18,"Neqa":19,"Neqb":20,"Nes":21,"Neu":22,"Nf":23,"Ng":24,"Nh":25,"Nv":26,"P":27,"T":28,"VA":29,"VAC":30,"VB":31,"VC":32,"VCL":33,"VD":34,"VF":35,"VE":36,"VG":37,"VH":38,"VHC":39,"VI":40,"VJ":41,"VK":42,"VL":43,"V_2":44,"DE":45,"SHI":46,"FW":47,"COLONCATEGORY":48,"COMMACATEGORY":49,"DASHCATEGORY":50,"DOTCATEGORY":51,"ETCCATEGORY":52,"EXCLAMATIONCATEGORY":53,"PARENTHESISCATEGORY":54,"PAUSECATEGORY":55,"PERIODCATEGORY":56,"QUESTIONCATEGORY":57,"SEMICOLONCATEGORY":58,"SPCHANGECATEGORY":59,"WHITESPACE":60,}
#fn="./files/is-topic_trainning_set/false.csv"
topics=[]
for path in paths:
    for file in listdir(path):
        topics.append(path + file) 
#topics=topics[:100]

# Testing Tool

#https://www.tutorialspoint.com/jupyter/jupyter_notebook_markdown_cells.htm

# tqdm example:
for i in tqdm(range(10)):
   somthing...
ref:https://stackoverflow.com/questions/3160699/python-progress-bar

# Multiprocess Functions

In [ ]:
import multiprocessing as mp
import time
from os.path import exists
import json
from MultiProcessSocket import MultiProcessSocket

In [ ]:
def add_word(dictionary,words):
    for w in words:
        if w in dictionary['word_indexs']:            
            index = dictionary['word_indexs'][w]
            dictionary['word_set'][index][1] += 1
        else:
            dictionary['word_indexs'][w] = dictionary['word_counter']
            dictionary['word_set'][dictionary['word_counter']][0] = w
            dictionary['word_set'][dictionary['word_counter']][1] = 1
            dictionary['word_counter'] += 1

In [ ]:
def worker(manager, path):
    with open(path, mode="r") as f: 
        jsonn = json.loads(f.read())
        words=[]
        try:
            for info in jsonn["content"]:
                doc = CkipDocument(raw = info)
                pipeline.get_ws(doc)
                words += flatten(doc.ws.to_list())
            manager.call(words)
        except Exception as e:
            print(e)
            print(path)

In [ ]:
def main(): 
    # "mp.cpu_count() + 2" Common practice to ensure that the pool will be saturated.
    pool = mp.Pool(mp.cpu_count()+2)    
    manager = MultiProcessSocket(
                add_word,
                {
                    'word_indexs':{},
                    'word_set':[["",0] for i in range(300000)],
                    'word_counter':0    
                }
              )
    manager.start()
    #pool.apply_async(s.start, (add_word,))
    progress = tqdm(total=len(topics))
    #fire off workers
    jobs = []
    for i in topics:
        job = pool.apply_async(worker, (manager,i))
        jobs.append(job)

    # collect results from the workers through the pool result queue
    for job in jobs:
        job.wait() 
        #job.get()
        progress.update(1)  
        
    #print("main:"+str(id(d.word_set))+"\n",end="")
    with open("files/topics_dictionary.txt","w",encoding="utf-8") as f:
        d = manager.get_data()
        dictionary = d['word_set'][:d['word_counter']]
        dictionary.sort(key=lambda x: x[1], reverse=True)        
        f.write(json.dumps(dictionary, indent=4, ensure_ascii=False))
        print("finish")
    manager.close()
    pool.close()
    pool.join()

In [ ]:
if __name__ == "__main__":
    main()